<a href="https://colab.research.google.com/github/mnehan67/flyrank-ml-internship-starter/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-10 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mnehan67/flyrank-ml-internship-starter/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

**Lane locked:** CTR / Engagement Opportunity Scoring  
**Development slice:** March 2026 (`month=2026-03`) — the same mid-panel month used in W03.

### Baseline in one sentence

**Review high-volume pages that rank well enough to earn clicks but have lower observed CTR than comparable pages in the same search-position bucket.**

The rule is deliberately simple and transparent. It is the baseline that the Week-5 model must beat; it does **not** claim that editing a page will cause traffic growth.


In [1]:
# Small Colab setup.
%pip -q install -U duckdb huggingface_hub pandas


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.5/21.5 MB 76.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 842.9/842.9 kB 56.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 106.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 3.0.6 which is incompatible.


In [2]:
import getpass
import json
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd
from huggingface_hub import HfApi

HF_TOKEN = getpass.getpass(
    "Paste your Hugging Face plain READ token (hidden; not saved): "
).strip()

if not HF_TOKEN.startswith("hf_"):
    raise RuntimeError(
        "That does not look like a Hugging Face token. "
        "Create a plain Read token and paste the full hf_... value."
    )

try:
    me = HfApi().whoami(token=HF_TOKEN)
    print(f"✓ Token valid for Hugging Face account: {me.get('name', 'unknown')}")
except Exception as e:
    raise RuntimeError(
        "The token is invalid. Create a NEW plain Read token, restart the runtime, "
        "and paste it into this hidden prompt."
    ) from e

con = duckdb.connect()
con.execute("SET enable_progress_bar = false")

safe_token = HF_TOKEN.replace("'", "''")
con.execute(
    f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{safe_token}')"
)

REL = "hf://datasets/FlyRank/internship-warehouse"
MAR = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"

march_rows = con.sql(f"SELECT COUNT(*) FROM {MAR}").fetchone()[0]
print(f"✓ March partition reachable: {march_rows:,} raw daily rows")


Paste your Hugging Face plain READ token (hidden; not saved): ··········
✓ Token valid for Hugging Face account: muhnehh
✓ March partition reachable: 9,841,378 raw daily rows


## 1. Check two signals first

My rule leans on two ideas:

1. **CTR vs. position — FlyRank flag-linked signal.** Pages at better search positions should normally capture more clicks, so a low CTR is only meaningful after adjusting for position.
2. **Volume — quick-win/confidence signal.** A CTR gap matters more when a page has enough impressions; very low-volume CTR can be dominated by denominator noise.

Both checks below print a bucket table with **n** before I build the rule. Each prints exactly one verdict: **CONFIRMED, OPPOSITE, MIXED, or FALSE**.


In [3]:
pages = con.sql(f"""
WITH agg AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS impressions_march,
        SUM(gsc_clicks) AS clicks_march,
        SUM(gsc_sum_position) AS sum_position_march,
        COUNT(*) FILTER (WHERE gsc_impressions > 0) AS days_with_impressions,
        STDDEV_POP(gsc_avg_position)
            FILTER (WHERE gsc_impressions > 0) AS position_volatility,
        MAX(gsc_impressions) AS max_daily_impressions
    FROM {MAR}
    WHERE gsc_data_available IS TRUE
    GROUP BY 1, 2
)
SELECT
    client_hash_id,
    content_hash_id,
    impressions_march,
    clicks_march,
    sum_position_march / NULLIF(impressions_march, 0) AS avg_position,
    100.0 * clicks_march / NULLIF(impressions_march, 0) AS observed_ctr_pp,
    days_with_impressions,
    COALESCE(position_volatility, 0.0) AS position_volatility,
    max_daily_impressions / NULLIF(impressions_march, 0) AS impression_spikiness
FROM agg
WHERE impressions_march >= 100
  AND sum_position_march > 0
""").df()

pages["position_bucket"] = pd.cut(
    pages["avg_position"],
    bins=[0, 3, 10, 20, 50, np.inf],
    labels=["1_top3", "2_pos4_10", "3_pos11_20", "4_pos21_50", "5_pos51_plus"],
    right=True,
)

pages["volume_bucket"] = pd.cut(
    pages["impressions_march"],
    bins=[99, 499, 999, 4999, np.inf],
    labels=["1_100_499", "2_500_999", "3_1k_4999", "4_5k_plus"],
    right=True,
)

print(f"Eligible March page-month rows: {len(pages):,}")
print(f"Clients represented: {pages['client_hash_id'].nunique():,}")
display(pages.head())


Eligible March page-month rows: 101,441
Clients represented: 44


,client_hash_id,content_hash_id,impressions_march,clicks_march,avg_position,observed_ctr_pp,days_with_impressions,position_volatility,impression_spikiness,position_bucket,volume_bucket
0,client_73cda7b4e4f265ea,content_7a105f548d9c6916,6523.0,7.0,6.893301,0.107313,31,2.402541,0.085084,2_pos4_10,4_5k_plus
1,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,453.0,0.0,3.214128,0.000000,31,2.440963,0.057395,2_pos4_10,1_100_499
2,client_73cda7b4e4f265ea,content_36c36abc7650d7af,5630.0,6.0,6.535346,0.106572,31,1.223326,0.060036,2_pos4_10,4_5k_plus
3,client_73cda7b4e4f265ea,content_a7da352b73b02668,4944.0,13.0,7.435680,0.262945,31,0.967035,0.057646,2_pos4_10,3_1k_4999
4,client_73cda7b4e4f265ea,content_1855a661b4d36130,429.0,1.0,3.871795,0.233100,31,4.270422,0.062937,2_pos4_10,1_100_499


In [4]:
# SIGNAL CHECK 1 — CTR vs search position (flag-linked)

signal1 = (
    pages.groupby("position_bucket", observed=True)
    .agg(
        n=("content_hash_id", "size"),
        median_ctr_pp=("observed_ctr_pp", "median"),
        mean_ctr_pp=("observed_ctr_pp", "mean"),
        median_impressions=("impressions_march", "median"),
    )
    .reset_index()
)

display(signal1.round(4))

page1_median = pages.loc[
    (pages["avg_position"] > 0) & (pages["avg_position"] <= 10),
    "observed_ctr_pp"
].median()

deeper_median = pages.loc[
    (pages["avg_position"] > 10) & (pages["avg_position"] <= 50),
    "observed_ctr_pp"
].median()

if pd.isna(page1_median) or pd.isna(deeper_median):
    verdict1 = "FALSE"
elif page1_median > deeper_median:
    verdict1 = "CONFIRMED"
elif page1_median < deeper_median:
    verdict1 = "OPPOSITE"
else:
    verdict1 = "MIXED"

print(f"Signal 1 verdict: {verdict1}")
print(
    f"Page-one median CTR = {page1_median:.4f} pp; "
    f"positions 11-50 median CTR = {deeper_median:.4f} pp."
)


,position_bucket,n,median_ctr_pp,mean_ctr_pp,median_impressions
0,1_top3,10194,0.2114,0.3400,1594.0
1,2_pos4_10,47811,0.1901,0.3197,1011.0
2,3_pos11_20,19547,0.1008,0.2442,564.0
3,4_pos21_50,19758,0.0000,0.1426,571.0
4,5_pos51_plus,4131,0.0000,0.0459,201.0


Signal 1 verdict: CONFIRMED
Page-one median CTR = 0.1941 pp; positions 11-50 median CTR = 0.0416 pp.


In [5]:
# SIGNAL CHECK 2 — volume as confidence / quick-win context

def q25(s):
    return s.quantile(0.25)

def q75(s):
    return s.quantile(0.75)

signal2 = (
    pages.groupby("volume_bucket", observed=True)
    .agg(
        n=("content_hash_id", "size"),
        median_ctr_pp=("observed_ctr_pp", "median"),
        zero_click_share=("clicks_march", lambda s: (s == 0).mean()),
        ctr_q25=("observed_ctr_pp", q25),
        ctr_q75=("observed_ctr_pp", q75),
    )
    .reset_index()
)

signal2["ctr_iqr_pp"] = signal2["ctr_q75"] - signal2["ctr_q25"]
display(signal2.round(4))

low_zero = signal2.loc[
    signal2["volume_bucket"].astype(str) == "1_100_499", "zero_click_share"
].iloc[0]
high_zero = signal2.loc[
    signal2["volume_bucket"].astype(str) == "4_5k_plus", "zero_click_share"
].iloc[0]

if high_zero < low_zero:
    verdict2 = "CONFIRMED"
elif high_zero > low_zero:
    verdict2 = "OPPOSITE"
else:
    verdict2 = "MIXED"

print(f"Signal 2 verdict: {verdict2}")
print(
    f"Zero-click share: 100-499 impressions = {low_zero:.2%}; "
    f"5k+ impressions = {high_zero:.2%}."
)


,volume_bucket,n,median_ctr_pp,zero_click_share,ctr_q25,ctr_q75,ctr_iqr_pp
0,1_100_499,39517,0.0000,0.6828,0.0000,0.3145,0.3145
1,2_500_999,16866,0.1425,0.3801,0.0000,0.3356,0.3356
2,3_1k_4999,31766,0.1898,0.1305,0.0751,0.4077,0.3326
3,4_5k_plus,13292,0.2041,0.0181,0.0931,0.3891,0.2960


Signal 2 verdict: CONFIRMED
Zero-click share: 100-499 impressions = 68.28%; 5k+ impressions = 1.81%.


### Rule reasoning after the signal checks

I keep the lane **CTR / Engagement Opportunity Scoring**.

My baseline uses the session's CTR-fix idea directly:

- compare a page's CTR only with pages in the **same position bucket**;
- require **at least 500 impressions** so the queue is not dominated by tiny denominators;
- focus on **average positions 1–20**, where snippet/title review is actionable;
- score only a **positive CTR gap**;
- scale the gap by impressions so a large, credible under-capture ranks above a tiny one.

There is **one reason code**: `high_volume_position_adjusted_ctr_gap`  
There is **one action label**: `review_title_meta_and_intent`


## 2. Encode ONE rule and write the ranked queue

For each position bucket:

`expected_ctr_pp = median observed CTR in that position bucket`

Then:

`ctr_gap_pp = expected_ctr_pp - observed_ctr_pp`

and the transparent baseline score is:

`baseline_action_score = max(ctr_gap_pp, 0) / 100 × impressions_march`

That final quantity is a **directional review score**, not guaranteed future clicks.


In [6]:
expected_ctr = (
    pages.groupby("position_bucket", observed=True)["observed_ctr_pp"]
    .median()
    .rename("expected_ctr_pp")
)

queue = pages.join(expected_ctr, on="position_bucket")
queue["ctr_gap_pp"] = queue["expected_ctr_pp"] - queue["observed_ctr_pp"]
queue["positive_ctr_gap_pp"] = queue["ctr_gap_pp"].clip(lower=0)

queue["eligible_for_review"] = (
    (queue["impressions_march"] >= 500)
    & (queue["avg_position"] > 0)
    & (queue["avg_position"] <= 20)
    & (queue["positive_ctr_gap_pp"] > 0)
)

queue["baseline_action_score"] = (
    queue["positive_ctr_gap_pp"] / 100.0 * queue["impressions_march"]
)

queue["reason_code"] = np.where(
    queue["eligible_for_review"],
    "high_volume_position_adjusted_ctr_gap",
    "not_selected",
)
queue["action_label"] = np.where(
    queue["eligible_for_review"],
    "review_title_meta_and_intent",
    "monitor",
)

ranked = (
    queue.loc[queue["eligible_for_review"]]
    .sort_values(
        ["baseline_action_score", "impressions_march"],
        ascending=[False, False],
    )
    .reset_index(drop=True)
)

ranked.insert(0, "rank", np.arange(1, len(ranked) + 1))

assert len(ranked) >= 10, "Fewer than 10 candidates; inspect thresholds before continuing."
assert (ranked["reason_code"] == "high_volume_position_adjusted_ctr_gap").all()
assert (ranked["action_label"] == "review_title_meta_and_intent").all()

print(f"Ranked review candidates: {len(ranked):,}")

display(
    ranked[
        [
            "rank",
            "client_hash_id",
            "content_hash_id",
            "baseline_action_score",
            "impressions_march",
            "avg_position",
            "observed_ctr_pp",
            "expected_ctr_pp",
            "ctr_gap_pp",
            "reason_code",
            "action_label",
        ]
    ].head(10).round(4)
)


Ranked review candidates: 22,185


,rank,client_hash_id,content_hash_id,baseline_action_score,impressions_march,avg_position,observed_ctr_pp,expected_ctr_pp,ctr_gap_pp,reason_code,action_label
0,1,client_23a62021009f63c4,content_44f34c0a90047651,425.0571,212404.0,0.6659,0.0113,0.2114,0.2001,high_volume_position_adjusted_ctr_gap,review_title_meta_and_intent
1,2,client_73cda7b4e4f265ea,content_8e1334d6356668e3,284.3784,134984.0,2.6930,0.0007,0.2114,0.2107,high_volume_position_adjusted_ctr_gap,review_title_meta_and_intent
2,3,client_73cda7b4e4f265ea,content_fec55986a1868d62,261.3150,124075.0,0.3084,0.0008,0.2114,0.2106,high_volume_position_adjusted_ctr_gap,review_title_meta_and_intent
3,4,client_62f4a7e64f5e0096,content_34a70fea29d15f24,228.8992,143019.0,3.1661,0.0301,0.1901,0.1600,high_volume_position_adjusted_ctr_gap,review_title_meta_and_intent
4,5,client_62f4a7e64f5e0096,content_f6116743b00afc2d,189.5323,107584.0,9.7357,0.0139,0.1901,0.1762,high_volume_position_adjusted_ctr_gap,review_title_meta_and_intent
5,6,client_73cda7b4e4f265ea,content_9c057b66c30a3abb,176.2389,83834.0,0.1160,0.0012,0.2114,0.2102,high_volume_position_adjusted_ctr_gap,review_title_meta_and_intent
6,7,client_62f4a7e64f5e0096,content_7c6373141eae744a,169.0779,132593.0,5.9485,0.0626,0.1901,0.1275,high_volume_position_adjusted_ctr_gap,review_title_meta_and_intent
7,8,client_9958f0a7ae1df715,content_cd3d932d4e1c8db0,165.8327,89332.0,7.8318,0.0045,0.1901,0.1856,high_volume_position_adjusted_ctr_gap,review_title_meta_and_intent
8,9,client_a80fca3f171ed1de,content_046fc480045b88f5,153.2928,83788.0,7.2083,0.0072,0.1901,0.1830,high_volume_position_adjusted_ctr_gap,review_title_meta_and_intent
9,10,client_a80fca3f171ed1de,content_9540d884af3e41fd,145.6084,82376.0,8.0052,0.0134,0.1901,0.1768,high_volume_position_adjusted_ctr_gap,review_title_meta_and_intent


In [7]:
OUT_DIR = Path("work/outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

csv_path = OUT_DIR / "baseline_action_score.csv"

export_cols = [
    "rank",
    "client_hash_id",
    "content_hash_id",
    "baseline_action_score",
    "impressions_march",
    "avg_position",
    "position_bucket",
    "observed_ctr_pp",
    "expected_ctr_pp",
    "ctr_gap_pp",
    "days_with_impressions",
    "position_volatility",
    "impression_spikiness",
    "reason_code",
    "action_label",
]

ranked[export_cols].to_csv(csv_path, index=False)

print(f"✓ Wrote {len(ranked):,} rows to {csv_path}")
print("CSV is intentionally not committed; the notebook regenerates it.")


✓ Wrote 22,185 rows to work/outputs/baseline_action_score.csv
CSV is intentionally not committed; the notebook regenerates it.


## 3. Top-10 review

For every top-ten row I record:

- **action** — what a reviewer should do;
- **why it is there** — the numbers that triggered the rule;
- **what would make it wrong** — the strongest plausible reason this could be a bad recommendation.

This is the skeptic check. The queue is not an automatic publishing instruction.


In [8]:
top10 = ranked.head(10).copy()

def why_line(r):
    return (
        f"{int(r.impressions_march):,} impressions; avg position {r.avg_position:.1f}; "
        f"CTR {r.observed_ctr_pp:.3f}% vs bucket median {r.expected_ctr_pp:.3f}% "
        f"(gap {r.ctr_gap_pp:.3f} pp)."
    )

p90_volatility = pages["position_volatility"].quantile(0.90)

def wrong_line(r):
    if r.impression_spikiness >= 0.25:
        return (
            "March visibility is spiky; one unusual day may inflate the apparent opportunity."
        )
    if r.days_with_impressions < 20:
        return (
            "The page was visible on relatively few March days, so the month may not represent normal demand."
        )
    if r.position_volatility >= p90_volatility:
        return (
            "Rank position was unusually volatile, so one monthly average may hide changing SERP exposure."
        )
    return (
        "Query mix, SERP features, brand/navigation intent, or seasonality could make its normal CTR "
        "lower than the position-bucket median."
    )

top10_review = pd.DataFrame({
    "rank": top10["rank"].astype(int),
    "action": top10["action_label"],
    "why_it_is_here": top10.apply(why_line, axis=1),
    "what_would_make_it_wrong": top10.apply(wrong_line, axis=1),
})

pd.set_option("display.max_colwidth", 160)
display(top10_review)

print("\nOne-line review:")
for _, r in top10_review.iterrows():
    print(
        f"#{r['rank']} — {r['action']} | "
        f"{r['why_it_is_here']} | WRONG IF: {r['what_would_make_it_wrong']}"
    )


,rank,action,why_it_is_here,what_would_make_it_wrong
0,1,review_title_meta_and_intent,"212,404 impressions; avg position 0.7; CTR 0.011% vs bucket median 0.211% (gap 0.200 pp).","Query mix, SERP features, brand/navigation intent, or seasonality could make its normal CTR lower than the position-bucket median."
1,2,review_title_meta_and_intent,"134,984 impressions; avg position 2.7; CTR 0.001% vs bucket median 0.211% (gap 0.211 pp).","Query mix, SERP features, brand/navigation intent, or seasonality could make its normal CTR lower than the position-bucket median."
2,3,review_title_meta_and_intent,"124,075 impressions; avg position 0.3; CTR 0.001% vs bucket median 0.211% (gap 0.211 pp).",March visibility is spiky; one unusual day may inflate the apparent opportunity.
3,4,review_title_meta_and_intent,"143,019 impressions; avg position 3.2; CTR 0.030% vs bucket median 0.190% (gap 0.160 pp).",March visibility is spiky; one unusual day may inflate the apparent opportunity.
4,5,review_title_meta_and_intent,"107,584 impressions; avg position 9.7; CTR 0.014% vs bucket median 0.190% (gap 0.176 pp).","Query mix, SERP features, brand/navigation intent, or seasonality could make its normal CTR lower than the position-bucket median."
5,6,review_title_meta_and_intent,"83,834 impressions; avg position 0.1; CTR 0.001% vs bucket median 0.211% (gap 0.210 pp).",March visibility is spiky; one unusual day may inflate the apparent opportunity.
6,7,review_title_meta_and_intent,"132,593 impressions; avg position 5.9; CTR 0.063% vs bucket median 0.190% (gap 0.128 pp).","Query mix, SERP features, brand/navigation intent, or seasonality could make its normal CTR lower than the position-bucket median."
7,8,review_title_meta_and_intent,"89,332 impressions; avg position 7.8; CTR 0.004% vs bucket median 0.190% (gap 0.186 pp).","Query mix, SERP features, brand/navigation intent, or seasonality could make its normal CTR lower than the position-bucket median."
8,9,review_title_meta_and_intent,"83,788 impressions; avg position 7.2; CTR 0.007% vs bucket median 0.190% (gap 0.183 pp).","Query mix, SERP features, brand/navigation intent, or seasonality could make its normal CTR lower than the position-bucket median."
9,10,review_title_meta_and_intent,"82,376 impressions; avg position 8.0; CTR 0.013% vs bucket median 0.190% (gap 0.177 pp).","Query mix, SERP features, brand/navigation intent, or seasonality could make its normal CTR lower than the position-bucket median."



One-line review:
#1 — review_title_meta_and_intent | 212,404 impressions; avg position 0.7; CTR 0.011% vs bucket median 0.211% (gap 0.200 pp). | WRONG IF: Query mix, SERP features, brand/navigation intent, or seasonality could make its normal CTR lower than the position-bucket median.
#2 — review_title_meta_and_intent | 134,984 impressions; avg position 2.7; CTR 0.001% vs bucket median 0.211% (gap 0.211 pp). | WRONG IF: Query mix, SERP features, brand/navigation intent, or seasonality could make its normal CTR lower than the position-bucket median.
#3 — review_title_meta_and_intent | 124,075 impressions; avg position 0.3; CTR 0.001% vs bucket median 0.211% (gap 0.211 pp). | WRONG IF: March visibility is spiky; one unusual day may inflate the apparent opportunity.
#4 — review_title_meta_and_intent | 143,019 impressions; avg position 3.2; CTR 0.030% vs bucket median 0.190% (gap 0.160 pp). | WRONG IF: March visibility is spiky; one unusual day may inflate the apparent opportunity.
#5 — r

## 4. Weak picks + leakage check

A good baseline should expose its own weaknesses. I use three diagnostics that were **not** part of the score — days with impressions, position volatility, and impression spikiness — to identify the top-ten row that looks least trustworthy.

### Leakage boundary

This rule uses only **March 2026 observed search measurements** to produce a **March review queue**. It does not touch April–June, the June `_sample`, a future label, product flags, raw URLs, raw queries, or client identity as a scoring feature.

`observed_ctr_pp` is a current observed signal used by the CTR-fix rule, not a future outcome fed into a predictive model.


In [9]:
top10_diag = top10.copy()

top10_diag["diagnostic_risk"] = (
    top10_diag["impression_spikiness"].rank(pct=True)
    + top10_diag["position_volatility"].rank(pct=True)
    + (-top10_diag["days_with_impressions"]).rank(pct=True)
)

weak = top10_diag.sort_values("diagnostic_risk", ascending=False).iloc[0]

print(f"Weakest-looking top-10 pick: rank #{int(weak['rank'])}")
print(
    f"days_with_impressions={int(weak['days_with_impressions'])}, "
    f"position_volatility={weak['position_volatility']:.3f}, "
    f"impression_spikiness={weak['impression_spikiness']:.3f}"
)
print(
    "Why it may be wrong: the rule only sees position-adjusted CTR gap × volume; "
    "it does not know the real query mix, SERP layout, seasonality, or snippet context."
)

FORBIDDEN_SCORING_FIELDS = {
    "client_hash_id",
    "content_hash_id",
    "future_ctr",
    "future_clicks",
    "trend_direction",
    "trend_pct",
    "health_score",
    "priority_score",
    "action_type",
}
RULE_SCORING_FIELDS = {
    "impressions_march",
    "avg_position",
    "position_bucket",
    "observed_ctr_pp",
    "expected_ctr_pp",
}

assert FORBIDDEN_SCORING_FIELDS.isdisjoint(RULE_SCORING_FIELDS)
assert ranked["rank"].is_monotonic_increasing
assert not ranked["baseline_action_score"].isna().any()

print("✓ Leakage check passed: no future-window, product-flag, or ID scoring inputs.")


Weakest-looking top-10 pick: rank #6
days_with_impressions=31, position_volatility=13.336, impression_spikiness=0.346
Why it may be wrong: the rule only sees position-adjusted CTR gap × volume; it does not know the real query mix, SERP layout, seasonality, or snippet context.
✓ Leakage check passed: no future-window, product-flag, or ID scoring inputs.


In [10]:
metrics = {
    "lane": "CTR / Engagement Opportunity Scoring",
    "development_month": "2026-03",
    "signal_1_ctr_vs_position_verdict": verdict1,
    "signal_2_volume_verdict": verdict2,
    "eligible_page_month_rows": int(len(pages)),
    "ranked_candidates": int(len(ranked)),
    "rule": (
        "position-bucket median CTR minus observed CTR, positive gap only; "
        ">=500 impressions; avg position 1-20; score = gap/100 * impressions"
    ),
    "reason_code": "high_volume_position_adjusted_ctr_gap",
    "action_label": "review_title_meta_and_intent",
}

json_path = OUT_DIR / "w04_baseline_metrics.json"
json_path.write_text(json.dumps(metrics, indent=2))

print(f"✓ Wrote metrics receipt: {json_path}")
display(pd.DataFrame([metrics]))


✓ Wrote metrics receipt: work/outputs/w04_baseline_metrics.json


,lane,development_month,signal_1_ctr_vs_position_verdict,signal_2_volume_verdict,eligible_page_month_rows,ranked_candidates,rule,reason_code,action_label
0,CTR / Engagement Opportunity Scoring,2026-03,CONFIRMED,CONFIRMED,101441,22185,"position-bucket median CTR minus observed CTR, positive gap only; >=500 impressions; avg position 1-20; score = gap/100 * impressions",high_volume_position_adjusted_ctr_gap,review_title_meta_and_intent


## 5. Self-check

After **Runtime → Run all**, confirm the outputs are visible and then save this notebook to your repo.

- [x] Lane is explicitly locked: CTR / Engagement Opportunity Scoring
- [x] Two signal checks are present
- [x] Both bucket tables print **n**
- [x] At least one signal is linked to a real FlyRank flag: CTR vs position
- [x] Each signal prints one verdict: CONFIRMED / OPPOSITE / MIXED / FALSE
- [x] Exactly one transparent baseline rule is encoded
- [x] Exactly one reason code is used for selected rows
- [x] Exactly one action label is used for selected rows
- [x] Ranked queue writes `work/outputs/baseline_action_score.csv`
- [x] Top ten each have action + why + what would make it wrong
- [x] Weak-pick check is shown
- [x] No future month, final-month `_sample`, product flag, or ID is used to score
- [ ] **Runtime → Run all** completes with no errors
- [ ] Executed notebook is saved as `work/notebooks/w04_baseline_score.ipynb`
- [ ] Commit the notebook (and optionally `work/outputs/w04_baseline_metrics.json`)
- [ ] Submit the repo URL

**Suggested commit:** `Complete W04 CTR baseline action score and top-10 review`
